In [1]:
import pickle as pkl
import pandas as pd #v3.0.2
import numpy as np # v1.26.4
import os
from collections import Counter
import plotly.express as px #plotly v6.6.0
from operator import methodcaller

In [2]:
directory = "datasets/MTBLS7337_152457/"
fileSamples = "s_MTBLS7337.txt"
fileData_PITC = "m_MTBLS7337_LC-MS_PITC_Panel_positive_reverse-phase_metabolite_profiling_v2_maf.tsv"
fileData_OrgAcids = "m_MTBLS7337_LC-MS_Organic_Acids_negative_reverse-phase_metabolite_profiling_v2_maf.tsv"
fileData_Lip1 = "m_MTBLS7337_DI-MS_Lipids_positive_1_metabolite_profiling_v2_maf.tsv"
fileData_Lip2 = "m_MTBLS7337_DI-MS_Lipids_positive_2_metabolite_profiling_v2_maf.tsv"

### Metadata

In [3]:
df_metadata = pd.read_csv(directory+fileSamples, delimiter="\t")
df_metadata = df_metadata.rename(columns={'Sample Name': 'Sample_Name'})
df_metadata

,Source Name,Characteristics[Organism],Term Source REF,Term Accession Number,Characteristics[Organism part],Term Source REF.1,Term Accession Number.1,Characteristics[Variant],Term Source REF.2,Term Accession Number.2,...,Term Accession Number.33,Factor Value[Other],Term Source REF.34,Term Accession Number.34,Factor Value[EQ-VAS],Term Source REF.35,Term Accession Number.35,Factor Value[Total SF-Yes2],Term Source REF.36,Term Accession Number.36
0,LC235,Homo sapiens,NCBITAXON,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,LC236,Homo sapiens,NCBITAXON,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,LC237,Homo sapiens,NCBITAXON,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,LC238,Homo sapiens,NCBITAXON,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,LC239,Homo sapiens,NCBITAXON,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,LC226,Homo sapiens,NCBITAXON,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,...,NaN,Yes,NaN,NaN,60.0,NaN,NaN,56.14438,NaN,NaN
258,LC228,Homo sapiens,NCBITAXON,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,...,NaN,No,NaN,NaN,35.0,NaN,NaN,88.80104,NaN,NaN
259,LC230,Homo sapiens,NCBITAXON,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,...,NaN,No,NaN,NaN,60.0,NaN,NaN,57.27021,NaN,NaN
260,LC232,Homo sapiens,NCBITAXON,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,...,NaN,No,NaN,NaN,70.0,NaN,NaN,113.32808,NaN,NaN


In [4]:
df_metadata["Sample_Name"]

0      LC235
1      LC236
2      LC237
3      LC238
4      LC239
       ...  
257    LC226
258    LC228
259    LC230
260    LC232
261    LC234
Name: Sample_Name, Length: 262, dtype: str

In [5]:
# df_metadata = df_metadata.rename(columns={'Sample Name': 'Sample_Name'})
# # Remove lines based on the Sample_Name column content (remove if NEG is present)
# df_metadata[~df_metadata.Sample_Name.str.contains("NEG")].to_csv(directory+"s_MTBLS28_POS.csv", sep=";")
# # Remove lines based on the Sample_Name column content (remove if POS is present)
# df_metadata[~df_metadata.Sample_Name.str.contains("POS")].to_csv(directory+"s_MTBLS28_NEG.csv", sep=";")


### Data

In [6]:
df_PITC = pd.read_csv(directory+fileData_PITC, delimiter="\t")
df_OrgAcids = pd.read_csv(directory+fileData_OrgAcids, delimiter="\t")
df_Lip1 = pd.read_csv(directory+fileData_Lip1, delimiter="\t")
df_Lip2 = pd.read_csv(directory+fileData_Lip2, delimiter="\t")

In [7]:
def preprocess_datamatrix(df, suffix:str, id_col="metabolite_identification"):
    metabo_names = df[id_col].apply(lambda x: x+suffix)
    df_data = df.filter(regex="^LC")  #keep only samples columns
    # if suffix == "__OrgAcids":
    #remove this column because of "Not Enough Volume" values, which is almost whole column in OrgAcids, and one value in Lip2, so we just remove the sample
    df_data = df_data.drop(columns=['LC177'])
    df_data[id_col] = metabo_names  #put back the names of metabolites to the matrix of samples
    df_data = df_data.set_index(id_col).T  #transpose to have samples as lines and features/metabolites as columns
    df_data.dropna(axis=1, how='all', inplace=True)  #drop columns where there is no values

    # Drops columns where more than 50% of values are "< LOD"
    df_data = df_data.loc[:, (df_data != "< LOD").mean() >= 0.5]
    
    
    #replace "< LOD" values with the minimum value of the same/current feature    
    for col_name, col_data in df_data.items():
        try:
            df_data.loc[df_data[col_name] == "< LOD", col_name] = col_data.min()
        except TypeError :
            df_data.loc[df_data[col_name] == "< LOD", col_name] = pd.to_numeric(col_data, errors='coerce').min() # convert all to floats to compute min (or coerce into NaN to skip str)

    return df_data

In [8]:
data_PITC = preprocess_datamatrix(df_PITC, "__PITC")
data_OrgAcids = preprocess_datamatrix(df_OrgAcids, "__OrgAcids")
data_Lip1 = preprocess_datamatrix(df_Lip1, "__Lip1")
data_Lip2 = preprocess_datamatrix(df_Lip2, "__Lip2")

/tmp/ipykernel_2221/1069211357.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_data[id_col] = metabo_names  #put back the names of metabolites to the matrix of samples


In [9]:
data_PITC

metabolite_identification,Acetyl-Ornithine__PITC,Alanine__PITC,alpha-Aminoadipic acid__PITC,alpha-Aminobutyric acid__PITC,Asparagine__PITC,Adenine__PITC,Adenosine__PITC,Allantoin__PITC,Aspartic acid__PITC,beta-Alanine__PITC,...,N-Acetyl-Alanine__PITC,N-Acetyl-Asparagine__PITC,N-Acetyl-Aspartic acid__PITC,N-Acetyl-Glutamic acid__PITC,N-Acetyl-Glycine__PITC,N-Acetyl-Histidine__PITC,N-Acetyl-Proline__PITC,N-Acetyl-Serine__PITC,N-Acetyl-Tryptophan__PITC,N-Acetyl-Valine__PITC
LC235,0.488,264.4,1.476,16.38,33.39,0.003,0.29,1.299,5.245,1.988,...,1.178,0.099,0.152,0.07,1.201,14.43,0.05,0.754,0.253,0.131
LC236,0.627,430.6,0.73,25.49,50.14,0.026,0.717,1.605,4.977,3.67,...,1.106,0.083,0.062,0.082,1.263,18.82,0.05,0.73,0.382,0.149
LC237,0.369,320.9,0.818,9.565,40.22,0.003,0.752,0.593,8.201,1.616,...,0.829,0.068,0.264,0.054,7.246,16.02,0.043,0.606,0.084,0.111
LC238,2.871,271,0.974,11.36,44.04,0.003,0.851,1.372,3.258,1.628,...,1.614,0.138,0.147,0.108,1.77,17.69,0.083,1.147,0.429,0.135
LC239,0.522,336.6,0.893,7.98,49.88,0.003,0.568,1.541,8.567,1.635,...,3.54,0.116,0.335,0.184,3.468,17.88,0.049,2.224,0.347,0.185
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LC226,0.539,357.6,2.862,14.65,29.85,0.077,0.761,1.216,4.988,3.19,...,0.813,0.051,0.116,0.138,1.554,22.85,0.039,0.914,0.518,0.067
LC228,0.58,564.3,3.552,7.889,82.17,0.105,0.81,2.469,30.56,2.917,...,3.933,0.304,0.573,0.268,2.538,12.85,0.097,3.134,1.758,0.07
LC230,0.549,353,1.776,12.25,51.09,0.021,0.77,1.607,9.039,1.853,...,1.093,0.111,0.242,0.102,1.87,23.26,0.085,1.432,0.75,0.063
LC232,0.724,489,1.949,16.82,56.95,0.049,1.098,1.873,6.683,5.683,...,2.172,0.112,0.46,0.188,4.392,30.18,0.072,1.634,0.59,0.176


In [10]:
# Concat the four data matrices to obtain one matrix for ML
# axis= 1 to join them horizontally (columns)
datamatrix = pd.concat([data_PITC, data_OrgAcids, data_Lip1, data_Lip2], axis=1, join='outer', ignore_index=False)

In [11]:
datamatrix

metabolite_identification,Acetyl-Ornithine__PITC,Alanine__PITC,alpha-Aminoadipic acid__PITC,alpha-Aminobutyric acid__PITC,Asparagine__PITC,Adenine__PITC,Adenosine__PITC,Allantoin__PITC,Aspartic acid__PITC,beta-Alanine__PITC,...,TG(20:5_36:3)__Lip2,TG(22:4_34:2)__Lip2,TG(22:5_32:0)__Lip2,TG(22:5_32:1)__Lip2,TG(22:5_34:1)__Lip2,TG(22:5_34:2)__Lip2,TG(22:6_32:0)__Lip2,TG(22:6_32:1)__Lip2,TG(22:6_34:1)__Lip2,TG(22:6_34:2)__Lip2
LC235,0.488,264.4,1.476,16.38,33.39,0.003,0.29,1.299,5.245,1.988,...,0.9385,1.3298,0.7596,0.0,7.8886,5.0973,5.6039,6.0233,28.5313,16.5848
LC236,0.627,430.6,0.73,25.49,50.14,0.026,0.717,1.605,4.977,3.67,...,1.307,1.423,1.0545,0.0,8.8948,4.6719,3.953,5.1645,21.7262,9.5373
LC237,0.369,320.9,0.818,9.565,40.22,0.003,0.752,0.593,8.201,1.616,...,0.8747,0.8711,0.378,0.0,4.0305,2.9784,1.1166,2.6484,4.9386,3.2678
LC238,2.871,271,0.974,11.36,44.04,0.003,0.851,1.372,3.258,1.628,...,1.4858,0.6765,0.8131,0.0,5.0834,2.7934,0.653,0.6265,3.3044,2.7373
LC239,0.522,336.6,0.893,7.98,49.88,0.003,0.568,1.541,8.567,1.635,...,0.1907,0.3728,0.0737,0.0,3.4047,1.1611,0.2287,0.0,1.2577,0.7242
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LC226,0.539,357.6,2.862,14.65,29.85,0.077,0.761,1.216,4.988,3.19,...,0.7209,4.0368,3.6541,2.5972,14.5767,7.0942,4.2672,8.1279,19.0716,7.5179
LC228,0.58,564.3,3.552,7.889,82.17,0.105,0.81,2.469,30.56,2.917,...,0.5655,0.52,0.7141,0.0,3.3448,1.2776,0.503,0.4544,3.5536,1.929
LC230,0.549,353,1.776,12.25,51.09,0.021,0.77,1.607,9.039,1.853,...,0.5235,0.7603,0.0,0.0,3.3245,3.1548,1.0928,1.9478,5.6929,5.3065
LC232,0.724,489,1.949,16.82,56.95,0.049,1.098,1.873,6.683,5.683,...,0.968,1.107,0.505,0.0,4.2416,3.0285,2.3182,1.651,11.5779,7.0282


In [22]:
for i, name in enumerate (datamatrix.columns):
    if "," in name:
        print(name)

2,5-Furandicarboxylic acid__OrgAcids


In [23]:
datamatrix.rename(columns={'2,5-Furandicarboxylic acid__OrgAcids': '2.5-Furandicarboxylic acid__OrgAcids'}, inplace=True)

In [25]:
# Save the df to csv file
datamatrix.to_csv("MTBLS7337_4DatamatricesCombined_forML.csv", sep=";")


In [17]:
#save metadata to csv to fit the pico format requirements
df_metadata = df_metadata[df_metadata['Source Name'] != "LC177"]
df_metadata.to_csv("MTBLS7337_metadata.csv", sep=";")

In [18]:
df_metadata

,Source Name,Characteristics[Organism],Term Source REF,Term Accession Number,Characteristics[Organism part],Term Source REF.1,Term Accession Number.1,Characteristics[Variant],Term Source REF.2,Term Accession Number.2,...,Term Accession Number.33,Factor Value[Other],Term Source REF.34,Term Accession Number.34,Factor Value[EQ-VAS],Term Source REF.35,Term Accession Number.35,Factor Value[Total SF-Yes2],Term Source REF.36,Term Accession Number.36
0,LC235,Homo sapiens,NCBITAXON,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,LC236,Homo sapiens,NCBITAXON,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,LC237,Homo sapiens,NCBITAXON,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,LC238,Homo sapiens,NCBITAXON,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,LC239,Homo sapiens,NCBITAXON,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,LC226,Homo sapiens,NCBITAXON,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,...,NaN,Yes,NaN,NaN,60.0,NaN,NaN,56.14438,NaN,NaN
258,LC228,Homo sapiens,NCBITAXON,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,...,NaN,No,NaN,NaN,35.0,NaN,NaN,88.80104,NaN,NaN
259,LC230,Homo sapiens,NCBITAXON,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,...,NaN,No,NaN,NaN,60.0,NaN,NaN,57.27021,NaN,NaN
260,LC232,Homo sapiens,NCBITAXON,http://purl.obolibrary.org/obo/NCBITaxon_9606,blood plasma,BTO,http://purl.obolibrary.org/obo/BTO_0000131,NaN,NaN,NaN,...,NaN,No,NaN,NaN,70.0,NaN,NaN,113.32808,NaN,NaN


In [1]:
# pd.read_csv("MTBLS7337_4DatamatricesCombined_forML.csv", delimiter=";")
# pd.read_csv("MTBLS7337_4DatamatricesCombined_forML.csv", sep=None, engine="python", index_col=0)